# Module 2: First Principles of Vector Search

Qdrant Beginners Course, follow-along notebook.

Course page: https://qdrant.tech/course/beginners/module-2/

## Recap: Module 1

Embeddings turn text into vectors. Cosine similarity measures how close two vectors are. Similarity alone misses word order, negation, and exact codes, that's what payload filters fix.

This module: collections, points, payloads, HNSW, filtering, chunking, and a full ingestion pipeline in Qdrant.


In [ ]:
!pip install -q "qdrant-client[fastembed]" 

## Core data model

- **Collection**: like a table. Fixed vector size and distance metric.
- **Point**: the atomic unit. ID + vector + optional payload.
- **Vector**: a list of numbers, an embedding represents meaning.
- **Payload**: JSON metadata used for filtering.

Connect to your cluster (URL and API key from Qdrant Cloud, see Module 0):


In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(
    url="https://xyz-example.eu-west-1-0.aws.cloud.qdrant.io",
    api_key="<your-api-key>",
)

### Create a collection

Vector size and distance metric come from your embedding model: 384 dims and cosine for all-MiniLM-L6-v2.


In [ ]:
client.create_collection(
    collection_name="articles",
    vectors_config=models.VectorParams(
        size=384,
        distance=models.Distance.COSINE,
    ),
)

### Insert a point

`upsert` inserts if the ID is new, updates if it already exists.


In [ ]:
from qdrant_client.models import PointStruct

client.upsert(
    collection_name="articles",
    points=[
        PointStruct(
            id=1,
            vector=[0.12, -0.87, 0.33] + [0.0] * 381,   # 384-dim embedding (placeholder)
            payload={
                "title": "Car Repair Guide",
                "category": "automotive",
                "year": 2024,
                "region": "EU",
            },
        )
    ],
)

## Distance metrics

| Metric | Notes |
|--------|-------|
| `models.Distance.COSINE` | Angle between vectors, robust to magnitude |
| `models.Distance.DOT` | Faster than cosine for unit-length vectors |
| `models.Distance.EUCLID` | Absolute distance, sensitive to magnitude |
| `models.Distance.MANHATTAN` | Sum of absolute differences |

## Top-K retrieval

Qdrant finds the K most similar points to a query vector.


In [ ]:
results = client.query_points(
    collection_name="articles",
    query=[0.12, -0.87, 0.33] + [0.0] * 381,   # your query vector (placeholder)
    limit=3,
)

for r in results.points:
    print(r.id, r.score, r.payload)

## HNSW

Brute-force search over millions of vectors is slow. Qdrant uses HNSW, a graph-based approximate nearest neighbor index: a layered graph where search enters at the top and narrows down layer by layer. Trades a small amount of recall for large speed gains. Defaults work well for most use cases.

## Payload filtering

Filters apply during HNSW traversal, not after.

| Condition | What it does |
|-----------|--------------|
| `must` | AND logic |
| `should` | OR logic |
| `must_not` | Exclude matches |
| `Range` | Numeric comparisons (gte, lte, gt, lt) |
| `Geo` | Radius or bounding box |


In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

results = client.query_points(
    collection_name="articles",
    query=[0.12, -0.87, 0.33] + [0.0] * 381,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="category",
                match=MatchValue(value="automotive")
            )
        ]
    ),
    limit=5,
)

For fields you filter often, create a payload index with `client.create_payload_index()`. It makes filtered queries faster, and Qdrant Cloud strict mode requires it.

## Chunking strategies

Embedding models have a max token limit (256 for all-MiniLM-L6-v2). Split long text before embedding.

| Strategy | How it works | Trade-off |
|----------|--------------|-----------|
| Fixed-Size | Split every N tokens | May cut sentences mid-thought |
| Semantic | New chunk on topic shift | Slower, needs a model |
| Sliding Window | Overlapping chunks | More storage, duplicate content |

## Full ingestion pipeline


In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url="https://xyz-example.eu-west-1-0.aws.cloud.qdrant.io",
    api_key="<your-api-key>",
)
# Don't hardcode credentials in real code, use env vars or a secrets manager.

In [ ]:
from qdrant_client import models

client.create_collection(
    collection_name="articles",
    vectors_config=models.VectorParams(
        size=384,
        distance=models.Distance.COSINE,
    ))

# Qdrant Cloud strict mode rejects filters on unindexed fields.
client.create_payload_index(
    collection_name="articles",
    field_name="category",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

In [ ]:
from qdrant_client.models import PointStruct
from fastembed import TextEmbedding

model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim

documents = [
    {"id": 1, "text": "Car repair guide",  "category": "automotive"},
    {"id": 2, "text": "How to cook pasta",  "category": "food"},
]

points = [
    PointStruct(
        id=doc["id"],
        vector=vector.tolist(),
        payload={"title": doc["text"], "category": doc["category"]},
    )
    for doc, vector in zip(documents, model.embed([d["text"] for d in documents]))
]
# upload_points handles batching/retries; upsert is better for single points.
client.upload_points(collection_name="articles", points=points)

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

query_text   = "automobile maintenance"
query_vector = list(model.embed([query_text]))[0].tolist()

results = client.query_points(
    collection_name="articles",
    query=query_vector,
    query_filter=Filter(
        must=[FieldCondition(key="category", match=MatchValue(value="automotive"))]
    ),
    limit=3,
)

for r in results.points:
    print(f"Score: {r.score:.3f}  |  {r.payload['title']}")

### Try it yourself

Add a third document with its own category, re-run the filtered query, confirm it's excluded when the category doesn't match.


In [ ]:
documents.append({"id": 3, "text": "Best hiking trails in Colorado", "category": "outdoors"})
# TODO: embed, upsert, and re-run the filtered query above


## Further reading

- [What Is HNSW](https://qdrant.tech/course/essentials/day-2/what-is-hnsw/)
- [Filtering](https://qdrant.tech/documentation/search/filtering/)
- [Payload Indexing](https://qdrant.tech/documentation/manage-data/indexing/#payload-index)
- [Chunking Strategies](https://qdrant.tech/course/essentials/day-1/chunking-strategies/)

Next: `Module3.ipynb`, sparse vectors and hybrid search.
